## 14. Isolation Forest (Unsupervised Anomaly Detection)

In [1]:
from sklearn.ensemble import IsolationForest
import numpy as np

# Use only numeric features (IF cannot handle category dtype)
numeric_features = [f for f in features
                    if str(X_train[f].dtype) not in ('category','object')]

X_train_if = X_train[numeric_features].fillna(X_train[numeric_features].median())
X_val_if   = X_val[numeric_features].fillna(X_train[numeric_features].median())

iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.035,   # ~fraud rate in dataset
    random_state=42,
    n_jobs=-1
)
iso_model.fit(X_train_if)

# Convert to [0,1] where 1 = most anomalous
raw_scores = iso_model.decision_function(X_val_if)
iso_scores = 1 - (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())

from sklearn.metrics import average_precision_score
print('Isolation Forest PR-AUC:', average_precision_score(X_val['isFraud'], iso_scores))
print('(Low is expected - IF is unsupervised, it adds diversity not raw score)')

NameError: name 'features' is not defined

## 15. 3-Model Weighted Ensemble

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

best_score   = -1
best_weights = (0.50, 0.35, 0.15)

for w_lgb in np.arange(0.30, 0.65, 0.05):
    for w_xgb in np.arange(0.25, 0.65, 0.05):
        w_iso = round(1.0 - w_lgb - w_xgb, 4)
        if not (0.0 <= w_iso <= 0.25):
            continue
        preds = w_lgb * lgb_preds + w_xgb * xgb_preds + w_iso * iso_scores
        score = average_precision_score(X_val['isFraud'], preds)
        if score > best_score:
            best_score   = score
            best_weights = (w_lgb, w_xgb, w_iso)

w_lgb, w_xgb, w_iso = best_weights
final_preds = w_lgb * lgb_preds + w_xgb * xgb_preds + w_iso * iso_scores
final_prauc = average_precision_score(X_val['isFraud'], final_preds)
iso_prauc   = average_precision_score(X_val['isFraud'], iso_scores)

print(f'LightGBM   PR-AUC : {lgb_prauc:.6f}')
print(f'XGBoost    PR-AUC : {xgb_prauc:.6f}')
print(f'IsoForest  PR-AUC : {iso_prauc:.6f}')
print(f'3-Ensemble PR-AUC : {final_prauc:.6f}')
print(f'Best weights : LGB={w_lgb:.2f}  XGB={w_xgb:.2f}  ISO={w_iso:.2f}')

## 16. Save All Models

In [ ]:
import joblib, os, json

joblib.dump(model,           'lgb_final.pkl')
joblib.dump(xgb_model,       'xgb_final.pkl')
joblib.dump(iso_model,       'iso_final.pkl')
joblib.dump(features,        'features_final.pkl')
joblib.dump(numeric_features,'numeric_features_final.pkl')

# Save training medians for API imputation
medians = X_train[numeric_features].median().to_dict()
with open('train_medians.json','w') as f:
    json.dump(medians, f)

for fname in ['lgb_final.pkl','xgb_final.pkl','iso_final.pkl',
              'features_final.pkl','numeric_features_final.pkl','train_medians.json']:
    size = os.path.getsize(fname)
    print(f'  {fname}: {size/1e6:.1f} MB')

---
## Phase 2 — Fraud Engine

### 17. AdaptiveThresholdEngine

In [ ]:
from collections import deque

class AdaptiveThresholdEngine:
    """
    Tracks rolling fraud rate over last N transactions.
    Tightens threshold when fraud rate spikes above expected.
    """
    def __init__(self,
                 base_threshold: float = 0.50,
                 expected_fraud_rate: float = 0.035,
                 window: int = 1000,
                 min_threshold: float = 0.25,
                 max_threshold: float = 0.70):
        self.base_threshold  = base_threshold
        self.expected_rate   = expected_fraud_rate
        self.window          = window
        self.min_threshold   = min_threshold
        self.max_threshold   = max_threshold
        self._recent_labels  = deque(maxlen=window)

    def update(self, is_fraud: int):
        self._recent_labels.append(is_fraud)

    @property
    def current_threshold(self) -> float:
        if len(self._recent_labels) < 50:
            return self.base_threshold
        observed_rate = sum(self._recent_labels) / len(self._recent_labels)
        ratio = observed_rate / max(self.expected_rate, 1e-6)
        if ratio > 2.0:   t = self.base_threshold * 0.70
        elif ratio > 1.5: t = self.base_threshold * 0.85
        elif ratio < 0.5: t = self.base_threshold * 1.15
        else:             t = self.base_threshold
        return float(np.clip(t, self.min_threshold, self.max_threshold))

    def should_flag(self, score: float) -> bool:
        return score >= self.current_threshold

    def __repr__(self):
        n    = len(self._recent_labels)
        rate = sum(self._recent_labels) / max(n, 1)
        return (f'AdaptiveThresholdEngine('
                f'threshold={self.current_threshold:.3f}, '
                f'observed_rate={rate:.3f}, window={n})')

# Tests
eng = AdaptiveThresholdEngine()
print('Fresh:          ', eng.current_threshold)
for _ in range(800): eng.update(0)
for _ in range(20):  eng.update(1)
print('Low fraud:      ', eng.current_threshold)
for _ in range(100): eng.update(1)
print('After spike:    ', eng.current_threshold)
print(eng)

### 18. Business Rules Layer

In [ ]:
class BusinessRulesLayer: 
    """
    Hard rules that fire regardless of ML score.
    Returns (flagged: bool, reasons: list[str])
    """
    HIGH_RISK_EMAILS = {
        'protonmail.com','mail.com','guerrillamail.com',
        'tempmail.com','throwam.com'
    }

    def evaluate(self, score, amount, hour,
                 email_domain='', adaptive_threshold=0.50):
        reasons = []
        flagged = False

        if amount > 50_000 and score > 0.30:
            reasons.append(f'HIGH_AMOUNT: {amount:,.0f} + score {score:.3f}')
            flagged = True

        if 2 <= hour <= 5 and score > 0.25:
            reasons.append(f'NIGHT_TXN: hour={hour}, score={score:.3f}')
            flagged = True

        if email_domain.lower() in self.HIGH_RISK_EMAILS and score > 0.20:
            reasons.append(f'RISKY_EMAIL: {email_domain}, score={score:.3f}')
            flagged = True

        if score >= adaptive_threshold:
            reasons.append(f'ML_SCORE: {score:.3f} >= threshold {adaptive_threshold:.3f}')
            flagged = True

        return flagged, reasons

rules = BusinessRulesLayer()
print('Large amount:', rules.evaluate(0.32, 75000, 14, 'gmail.com'))
print('Night txn:   ', rules.evaluate(0.28, 5000,  3,  'gmail.com'))
print('Risky email: ', rules.evaluate(0.22, 5000,  14, 'protonmail.com'))
print('Clean:       ', rules.evaluate(0.15, 500,   14, 'gmail.com'))

### 19. ScoredTransaction Dataclass

In [ ]:
from dataclasses import dataclass, field
import time

@dataclass
class ScoredTransaction:
    transaction_id: str
    lgb_score:      float
    xgb_score:      float
    iso_score:      float
    ensemble_score: float
    threshold:      float
    is_flagged:     bool
    reasons:        list
    latency_ms:     float = 0.0
    timestamp:      float = field(default_factory=time.time)

    def summary(self) -> str:
        flag = '🚨 FRAUD' if self.is_flagged else '✅ LEGIT'
        return (f'{flag} | txn={self.transaction_id} | '
                f'score={self.ensemble_score:.3f} | '
                f'threshold={self.threshold:.3f} | '
                f'latency={self.latency_ms:.1f}ms | '
                f'reasons={self.reasons}')

# Test
txn = ScoredTransaction(
    transaction_id='TXN_001', lgb_score=0.72, xgb_score=0.68,
    iso_score=0.55, ensemble_score=0.69, threshold=0.50,
    is_flagged=True,
    reasons=['ML_SCORE: 0.69 >= threshold 0.50', 'NIGHT_TXN: hour=3'],
    latency_ms=4.2
)
print(txn.summary())

### 20. Single-Transaction Inference Pipeline

In [ ]:
import time

_lgb         = joblib.load('lgb_finals.pkl')
_xgb         = joblib.load('xgb_finals.pkl')
_iso         = joblib.load('iso_finals.pkl')
_feats       = joblib.load('features_finals.pkl')
_num_feats   = joblib.load('numeric_features_finals.pkl')
_train_meds  = X_train[_num_feats].median()

_engine      = AdaptiveThresholdEngine()
_rules       = BusinessRulesLayer()

def score_transaction(row: dict) -> ScoredTransaction:
    t0  = time.perf_counter()
    df  = pd.DataFrame([row])

    # Align category dtypes
    for col in _feats:
        if col in df.columns and col in X_train.columns:
            if str(X_train[col].dtype) == 'category':
                df[col] = pd.Categorical(df[col],
                              categories=X_train[col].cat.categories)

    lgb_s = float(_lgb.predict_proba(df[_feats])[:, 1][0])
    xgb_s = float(_xgb.predict_proba(df[_feats])[:, 1][0])

    df_if = df[_num_feats].fillna(_train_meds)
    raw   = _iso.decision_function(df_if)[0]
    iso_s = float(np.clip(1 - (raw + 0.5), 0, 1))

    ens_s     = w_lgb * lgb_s + w_xgb * xgb_s + w_iso * iso_s
    threshold = _engine.current_threshold
    amount    = row.get('TransactionAmt', 0)
    hour      = int((row.get('TransactionDT', 0) // 3600) % 24)
    email     = str(row.get('P_emaildomain', ''))

    flagged, reasons = _rules.evaluate(
        score=ens_s, amount=amount, hour=hour,
        email_domain=email, adaptive_threshold=threshold
    )
    latency = (time.perf_counter() - t0) * 1000

    return ScoredTransaction(
        transaction_id=str(row.get('TransactionID','?')),
        lgb_score=lgb_s, xgb_score=xgb_s, iso_score=iso_s,
        ensemble_score=ens_s, threshold=threshold,
        is_flagged=flagged, reasons=reasons, latency_ms=latency
    )

# Test
result = score_transaction(X_val.iloc[0].to_dict())
print(result.summary())

### 21. Batch Throughput Benchmark

In [ ]:
import time, numpy as np

N = 1000
rows = X_val.head(N).to_dict(orient='records')

t0      = time.perf_counter()
results = [score_transaction(r) for r in rows]
elapsed = time.perf_counter() - t0

lats    = [r.latency_ms for r in results]
flagged = sum(r.is_flagged for r in results)

print(f'Scored {N} transactions in {elapsed:.2f}s')
print(f'Throughput  : {N/elapsed:,.0f} TPS')
print(f'Mean latency: {np.mean(lats):.2f} ms')
print(f'P50 latency : {np.percentile(lats,50):.2f} ms')
print(f'P95 latency : {np.percentile(lats,95):.2f} ms')
print(f'P99 latency : {np.percentile(lats,99):.2f} ms')
print(f'Flagged     : {flagged}/{N} ({100*flagged/N:.1f}%)')

---
## Phase 3 — Async Streaming Engine

### 22. AsyncIO Producer / Consumer

In [ ]:
import asyncio, time, numpy as np
from collections import deque

class StreamMonitor:
    def __init__(self, window=500):
        self._lats     = deque(maxlen=window)
        self._flags    = deque(maxlen=window)
        self.total     = 0
        self.n_flagged = 0
        self._t0       = time.perf_counter()

    def record(self, r: ScoredTransaction):
        self._lats.append(r.latency_ms)
        self._flags.append(int(r.is_flagged))
        self.total     += 1
        self.n_flagged += int(r.is_flagged)

    @property
    def tps(self):
        return self.total / max(time.perf_counter() - self._t0, 1e-6)

    @property
    def live_fraud_rate(self):
        return sum(self._flags) / max(len(self._flags), 1)

    @property
    def p95_latency(self):
        return float(np.percentile(list(self._lats), 95)) if self._lats else 0

    def summary(self):
        return (f'TPS={self.tps:,.0f} | '
                f'fraud_rate={self.live_fraud_rate:.3f} | '
                f'p95={self.p95_latency:.1f}ms | '
                f'total={self.total}')


async def producer(q, rows, speed=500):
    delay = 1.0 / speed
    for row in rows:
        await q.put(row)
        await asyncio.sleep(delay)
    await q.put(None)

async def consumer(q, monitor):
    flagged_log = []
    while True:
        row = await q.get()
        if row is None: break
        r = score_transaction(row)
        monitor.record(r)
        if r.is_flagged: flagged_log.append(r)
        q.task_done()
    return flagged_log

async def run_stream(rows, speed=500):
    q = asyncio.Queue(maxsize=200)
    m = StreamMonitor()
    prod = asyncio.create_task(producer(q, rows, speed))
    cons = asyncio.create_task(consumer(q, m))
    await prod
    flagged = await cons
    return m, flagged

# Run
stream_rows = X_val.head(2000).to_dict(orient='records')
monitor, flagged_txns = await run_stream(stream_rows, speed=500)

print('=' * 50)
print('STREAMING BENCHMARK')
print('=' * 50)
print(f'Transactions : {monitor.total:,}')
print(f'Throughput   : {monitor.tps:,.0f} TPS')
print(f'Flagged      : {monitor.n_flagged} ({100*monitor.n_flagged/monitor.total:.1f}%)')
print(f'P95 latency  : {monitor.p95_latency:.1f} ms')
print()
for t in flagged_txns[:3]:
    print(' ', t.summary())

---
## Phase 4 — FastAPI Backend

### 23. Write app.py

In [ ]:
app_code = 'import asyncio, sqlite3, time, json\nimport joblib, numpy as np, pandas as pd\nimport uvicorn\nfrom fastapi import FastAPI, WebSocket, WebSocketDisconnect\nfrom pydantic import BaseModel\n\n# ── Load models ────────────────────────────────────────────────────────────────\n_lgb        = joblib.load("lgb_final.pkl")\n_xgb        = joblib.load("xgb_final.pkl")\n_iso        = joblib.load("iso_final.pkl")\n_feats      = joblib.load("features_final.pkl")\n_num_feats  = joblib.load("numeric_features_final.pkl")\nwith open("train_medians.json") as f:\n    _meds = pd.Series(json.load(f))\n\nW_LGB, W_XGB, W_ISO = 0.50, 0.35, 0.15\nDB = "flagged.db"\n\ndef init_db():\n    con = sqlite3.connect(DB)\n    con.execute("CREATE TABLE IF NOT EXISTS flagged (id INTEGER PRIMARY KEY AUTOINCREMENT, txn_id TEXT, score REAL, threshold REAL, reasons TEXT, latency_ms REAL, reviewed INTEGER DEFAULT 0, label TEXT, ts REAL)")\n    con.commit(); con.close()\ninit_db()\n\nfrom collections import deque\n\nclass _ThresholdEngine:\n    def __init__(self):\n        self._buf = deque(maxlen=1000)\n    def update(self, v): self._buf.append(v)\n    @property\n    def threshold(self):\n        if len(self._buf) < 50: return 0.50\n        r = sum(self._buf)/len(self._buf)/0.035\n        if r > 2.0:   return 0.35\n        elif r > 1.5: return 0.42\n        elif r < 0.5: return 0.58\n        return 0.50\n\nclass _Monitor:\n    def __init__(self):\n        self.total=0; self.flagged=0\n        self._lats=deque(maxlen=500); self._t0=time.perf_counter()\n    def record(self, lat, flag):\n        self.total+=1; self.flagged+=flag; self._lats.append(lat)\n    @property\n    def tps(self): return self.total/max(time.perf_counter()-self._t0,1e-6)\n    @property\n    def p95(self): return float(np.percentile(list(self._lats),95)) if self._lats else 0\n\n_eng = _ThresholdEngine()\n_mon = _Monitor()\n_ws_clients = []\n\nHIGH_RISK = {\'protonmail.com\',\'mail.com\',\'guerrillamail.com\'}\n\napp = FastAPI(title="Fraud Detection Engine", version="1.0")\n\nclass TxnIn(BaseModel):\n    TransactionID: str = "?"\n    TransactionAmt: float = 0\n    TransactionDT: int = 0\n    P_emaildomain: str = ""\n    model_config = {"extra":"allow"}\n\nclass ReviewIn(BaseModel):\n    label: str\n\ndef _score(row):\n    t0  = time.perf_counter()\n    df  = pd.DataFrame([row])\n    lgb_s = float(_lgb.predict_proba(df[_feats])[:,1][0])\n    xgb_s = float(_xgb.predict_proba(df[_feats])[:,1][0])\n    raw   = _iso.decision_function(df[_num_feats].fillna(_meds))[0]\n    iso_s = float(np.clip(1-(raw+0.5),0,1))\n    ens   = W_LGB*lgb_s + W_XGB*xgb_s + W_ISO*iso_s\n    thr   = _eng.threshold\n    amt   = row.get("TransactionAmt",0)\n    hr    = int((row.get("TransactionDT",0)//3600)%24)\n    em    = str(row.get("P_emaildomain","")).lower()\n    reasons=[]\n    flag=False\n    if amt>50000 and ens>0.30: reasons.append(f"HIGH_AMOUNT:{amt:.0f}"); flag=True\n    if 2<=hr<=5 and ens>0.25:  reasons.append(f"NIGHT_TXN:hour={hr}"); flag=True\n    if em in HIGH_RISK and ens>0.20: reasons.append(f"RISKY_EMAIL:{em}"); flag=True\n    if ens>=thr: reasons.append(f"ML_SCORE:{ens:.3f}"); flag=True\n    lat=(time.perf_counter()-t0)*1000\n    return {"txn_id":row.get("TransactionID","?"),"lgb":lgb_s,"xgb":xgb_s,\n            "iso":iso_s,"score":ens,"threshold":thr,"flagged":flag,\n            "reasons":reasons,"latency_ms":lat}\n\n@app.post("/score")\nasync def score(txn: TxnIn):\n    r = _score(txn.model_dump())\n    _mon.record(r["latency_ms"], r["flagged"])\n    if r["flagged"]:\n        con=sqlite3.connect(DB)\n        con.execute("INSERT INTO flagged(txn_id,score,threshold,reasons,latency_ms,ts) VALUES(?,?,?,?,?,?)",\n                    (r["txn_id"],r["score"],r["threshold"],str(r["reasons"]),r["latency_ms"],time.time()))\n        con.commit(); con.close()\n        for ws in _ws_clients:\n            try: await ws.send_json(r)\n            except: pass\n    return r\n\n@app.get("/transactions/flagged")\nasync def get_flagged(limit:int=50, offset:int=0):\n    con=sqlite3.connect(DB)\n    rows=con.execute("SELECT * FROM flagged ORDER BY ts DESC LIMIT ? OFFSET ?",(limit,offset)).fetchall()\n    con.close()\n    cols=["id","txn_id","score","threshold","reasons","latency_ms","reviewed","label","ts"]\n    return [dict(zip(cols,r)) for r in rows]\n\n@app.post("/transactions/{txn_id}/review")\nasync def review(txn_id:str, body:ReviewIn):\n    con=sqlite3.connect(DB)\n    con.execute("UPDATE flagged SET reviewed=1,label=? WHERE txn_id=?",(body.label,txn_id))\n    con.commit(); con.close()\n    _eng.update(1 if body.label=="fraud" else 0)\n    return {"status":"ok"}\n\n@app.get("/metrics/live")\nasync def metrics():\n    return {"tps":round(_mon.tps,1),"total":_mon.total,\n            "flagged":_mon.flagged,"p95_ms":round(_mon.p95,2),\n            "threshold":round(_eng.threshold,4)}\n\n@app.websocket("/ws/live-flags")\nasync def ws_live(ws:WebSocket):\n    await ws.accept(); _ws_clients.append(ws)\n    try:\n        while True: await ws.receive_text()\n    except WebSocketDisconnect: _ws_clients.remove(ws)\n\nif __name__=="__main__":\n    uvicorn.run("app:app",host="0.0.0.0",port=8000,reload=False)\n'

with open("app.py","w") as f:
    f.write(app_code)

print("app.py written.")
print("Run with: pip install fastapi uvicorn && python app.py")
print("Endpoints:")
print("  POST /score              — score a transaction")
print("  GET  /transactions/flagged — list flagged txns")
print("  POST /transactions/{id}/review — analyst review")
print("  GET  /metrics/live       — live TPS + latency")
print("  WS   /ws/live-flags      — real-time push")

---
## Phase 5 — SHAP Explainability

### 24. SHAP Feature Contributions

In [ ]:
import shap

# TreeExplainer on LGB (fastest for tree models)
explainer = shap.TreeExplainer(model)
sample_df  = X_val[features].head(200)
shap_vals  = explainer.shap_values(sample_df)
sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals
print('SHAP values computed. Shape:', sv.shape)

def explain_row(row_df, top_n=3):
    vals = explainer.shap_values(row_df)
    sv   = vals[1][0] if isinstance(vals, list) else vals[0]
    pairs = sorted(zip(features, sv), key=lambda x: abs(x[1]), reverse=True)
    return [{'feature':f,'value':float(row_df[f].iloc[0]),
             'shap':round(float(s),4),
             'direction':'up fraud' if s>0 else 'down fraud'}
            for f,s in pairs[:top_n]]

# Demo on a high-score transaction
high_score_idx = final_preds.argmax()
row_df = X_val[features].iloc[[high_score_idx]].reset_index(drop=True)
expl = explain_row(row_df)
print(f'Transaction score: {final_preds[high_score_idx]:.4f}')
print('Top SHAP contributors:')
for e in expl:
    print(f"  {e['feature']:20s}  val={e['value']:.3f}  "
          f"shap={e['shap']:+.4f}  {e['direction']}")

---
## Phase 6 — Final Benchmark Report

### 25. All Metrics

In [ ]:
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    f1_score, precision_score, recall_score, precision_recall_curve
)
import numpy as np, time

y_true = X_val['isFraud'].values

# Full val iso scores (properly normalised)
X_val_if_full = X_val[numeric_features].fillna(X_train[numeric_features].median())
raw_full = iso_model.decision_function(X_val_if_full)
iso_full = 1 - (raw_full - raw_full.min()) / (raw_full.max() - raw_full.min())
final_full = w_lgb * lgb_preds + w_xgb * xgb_preds + w_iso * iso_full

# Optimal threshold from PR curve
prec, rec, threshs = precision_recall_curve(y_true, final_full)
f1_arr = 2*prec*rec / np.maximum(prec+rec, 1e-8)
opt_idx   = np.argmax(f1_arr)
opt_thresh = threshs[min(opt_idx, len(threshs)-1)]
y_pred    = (final_full >= opt_thresh).astype(int)

# Latency benchmark
N_BENCH = 500
rows_b = X_val.head(N_BENCH).to_dict(orient='records')
t0 = time.perf_counter()
res_b = [score_transaction(r) for r in rows_b]
elapsed = time.perf_counter() - t0
lats_b = [r.latency_ms for r in res_b]

print('=' * 55)
print('FINAL BENCHMARK REPORT')
print('=' * 55)
print(f'PR-AUC          : {average_precision_score(y_true, final_full):.4f}')
print(f'ROC-AUC         : {roc_auc_score(y_true, final_full):.4f}')
print(f'F1 (opt thresh) : {f1_score(y_true, y_pred):.4f}  @ {opt_thresh:.3f}')
print(f'Precision       : {precision_score(y_true, y_pred):.4f}')
print(f'Recall          : {recall_score(y_true, y_pred):.4f}')
print()
print(f'Throughput      : {N_BENCH/elapsed:,.0f} TPS')
print(f'Mean latency    : {np.mean(lats_b):.2f} ms')
print(f'P50 latency     : {np.percentile(lats_b,50):.2f} ms')
print(f'P95 latency     : {np.percentile(lats_b,95):.2f} ms')
print(f'P99 latency     : {np.percentile(lats_b,99):.2f} ms')
print()
print(f'Features        : {len(features)}')
print(f'LGB best iter   : {model.best_iteration_}')
print(f'XGB best iter   : {xgb_model.best_iteration}')
print(f'Ensemble weights: LGB={w_lgb:.2f} XGB={w_xgb:.2f} ISO={w_iso:.2f}')